# Integrated arXiv Search with Transformers

This notebook combines the arXiv data processing workflow with transformer-based embeddings to create a powerful semantic search system for research papers.

## 1. Setup and Dependencies

In [1]:
# Install required packages
!pip install requests d6tflow pandas numpy sentence-transformers torch transformers scikit-learn


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Import all required libraries
import os
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import d6tflow
import requests
import xml.etree.ElementTree as ET
from io import StringIO
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import json
import pickle

Welcome to d6tflow! For Q&A see https://github.com/d6t/d6tflow


## 2. Data Collection and Processing Tasks

In [3]:
# Configuration parameters
query_words = ['machine', 'learning', 'neural', 'network', 'transformer', 'attention']
query_size = 5000  # Increased for more comprehensive results

# Build query
queries = [word + '&' for word in query_words]
query = ''.join(queries)
url = f'https://export.arxiv.org/api/query?search_query=all:{query}start=0&max_results={query_size}'
print(f"Query URL: {url}")

# Search phrase for filtering
search_phrase = ' '.join(query_words[:2])  # "machine learning"
print(f"Search phrase: {search_phrase}")

Query URL: https://export.arxiv.org/api/query?search_query=all:machine&learning&neural&network&transformer&attention&start=0&max_results=5000
Search phrase: machine learning


In [4]:
class MetaDataTask(d6tflow.tasks.TaskPickle):
    """Task to fetch and store raw arXiv data"""
    
    def run(self):
        print("Fetching data from arXiv...")
        response = requests.get(url)
        data = response.text
        
        archive = {
            'url': url,
            'data': data,
            'fetch_time': datetime.now().isoformat()
        }
        
        result = {'meta': archive}
        self.save(result)
        print("Raw data saved successfully!")

In [5]:
class ProcessedDataTask(d6tflow.tasks.TaskPickle):
    """Task to process and clean arXiv data"""
    
    def requires(self):
        return MetaDataTask()
    
    def run(self):
        print("Processing arXiv data...")
        
        # Load raw data
        raw_data = self.input().load()
        xml_data = raw_data['meta']['data']
        
        # Parse XML data
        df = pd.read_xml(StringIO(xml_data) if isinstance(xml_data, str) else xml_data)
        
        # Create processed dataframe
        processed_df = pd.DataFrame()
        
        # Extract relevant fields (skip first 7 entries which are metadata)
        if len(df) > 7:
            processed_df['title'] = df['title'][7:].reset_index(drop=True)
            processed_df['abstract'] = df['summary'][7:].reset_index(drop=True)
            processed_df['published'] = pd.to_datetime(df['published'][7:].reset_index(drop=True))
            processed_df['updated'] = pd.to_datetime(df['updated'][7:].reset_index(drop=True))
            processed_df['url'] = df['id'][7:].reset_index(drop=True)
            
            # Add filtering columns
            two_years_ago = pd.Timestamp.now(tz='UTC') - pd.DateOffset(years=2)
            processed_df['is_recent'] = processed_df['published'].apply(lambda x: x > two_years_ago)
            processed_df['title_has_keywords'] = processed_df['title'].str.contains(search_phrase, case=False, na=False)
            processed_df['combined_text'] = processed_df['title'] + ' ' + processed_df['abstract']
            
            # Clean up any NaN values
            processed_df = processed_df.dropna(subset=['title', 'abstract'])
            
            print(f"Processed {len(processed_df)} papers")
            print(f"Recent papers (last 2 years): {processed_df['is_recent'].sum()}")
            print(f"Papers with keywords in title: {processed_df['title_has_keywords'].sum()}")
        
        else:
            print("Warning: Not enough data entries found")
            processed_df = pd.DataFrame()  # Empty dataframe
        
        self.save(processed_df)
        print("Processed data saved successfully!")

In [6]:
class EmbeddingTask(d6tflow.tasks.TaskPickle):
    """Task to generate embeddings for papers using transformer models"""
    
    def requires(self):
        return ProcessedDataTask()
    
    def run(self):
        print("Loading transformer model...")
        model = SentenceTransformer('paraphrase-albert-small-v2')
        
        # Load processed data
        processed_df = self.input().load()
        
        if len(processed_df) > 0:
            print(f"Generating embeddings for {len(processed_df)} papers...")
            
            # Generate embeddings for combined text (title + abstract)
            texts = processed_df['combined_text'].tolist()
            embeddings = model.encode(texts, show_progress_bar=True)
            
            # Save embeddings and associated metadata
            result = {
                'embeddings': embeddings,
                'papers': processed_df.to_dict('records'),
                'model_name': 'paraphrase-albert-small-v2',
                'embedding_dim': embeddings.shape[1]
            }
            
            print(f"Generated embeddings with shape: {embeddings.shape}")
        else:
            print("No data to process")
            result = {
                'embeddings': np.array([]),
                'papers': [],
                'model_name': 'paraphrase-albert-small-v2',
                'embedding_dim': 0
            }
        
        self.save(result)
        print("Embeddings saved successfully!")

## 3. Execute the Data Pipeline

In [7]:
# Run the complete pipeline
flow = d6tflow.Workflow()
result = flow.run(EmbeddingTask)
print("\n" + "="*50)
print("Pipeline execution completed!")
print("="*50)


===== Luigi Execution Summary =====

Scheduled 1 tasks of which:
* 1 complete ones were encountered:
    - 1 EmbeddingTask()

Did not run any tasks
This progress looks :) because there were no failed tasks or missing dependencies

===== Luigi Execution Summary =====


Pipeline execution completed!


## 4. Semantic Search Functionality

In [8]:
class SemanticSearch:
    """Semantic search functionality using transformer embeddings"""
    
    def __init__(self):
        # Load the embedding task results
        embedding_task = EmbeddingTask()
        self.data = embedding_task.load()
        
        self.embeddings = self.data['embeddings']
        self.papers = self.data['papers']
        self.model = SentenceTransformer(self.data['model_name'])
        
        print(f"Loaded {len(self.papers)} papers with {self.data['embedding_dim']}-dimensional embeddings")
    
    def search(self, query, top_k=10):
        """Search for papers similar to the query"""
        if len(self.papers) == 0:
            print("No papers available for search")
            return []
        
        # Generate embedding for query
        query_embedding = self.model.encode([query])
        
        # Calculate similarities
        similarities = cosine_similarity(query_embedding, self.embeddings)[0]
        
        # Get top k results
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            paper = self.papers[idx].copy()
            paper['similarity_score'] = similarities[idx]
            results.append(paper)
        
        return results
    
    def display_results(self, results, max_abstract_length=200):
        """Display search results in a readable format"""
        for i, paper in enumerate(results, 1):
            print(f"\n{i}. {paper['title']}")
            print(f"   Similarity: {paper['similarity_score']:.3f}")
            print(f"   Published: {paper['published'][:10]}")
            
            # Truncate abstract if too long
            abstract = paper['abstract']
            if len(abstract) > max_abstract_length:
                abstract = abstract[:max_abstract_length] + "..."
            print(f"   Abstract: {abstract}")
            print(f"   URL: {paper['url']}")
            print("-" * 80)

In [9]:
# Initialize semantic search
search_engine = SemanticSearch()

AttributeError: 'EmbeddingTask' object has no attribute 'load'

## 5. Interactive Search Examples

In [10]:
# Example searches
search_queries = [
    "deep learning for natural language processing",
    "computer vision and image recognition",
    "reinforcement learning algorithms",
    "transformer architecture attention mechanisms",
    "generative adversarial networks"
]

for query in search_queries:
    print(f"\n{'='*60}")
    print(f"SEARCH QUERY: {query}")
    print(f"{'='*60}")
    
    results = search_engine.search(query, top_k=3)
    search_engine.display_results(results)


SEARCH QUERY: deep learning for natural language processing


NameError: name 'search_engine' is not defined

## 6. Custom Search Interface

In [11]:
def interactive_search():
    """Interactive search function for custom queries"""
    print("\n🔍 ArXiv Semantic Search Interface")
    print("Enter your search query (or 'quit' to exit):")
    
    while True:
        query = input("\n> ").strip()
        
        if query.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        if not query:
            print("Please enter a search query.")
            continue
        
        print(f"\nSearching for: '{query}'...")
        results = search_engine.search(query, top_k=5)
        
        if results:
            search_engine.display_results(results)
        else:
            print("No results found.")

# Uncomment the next line to start interactive search
# interactive_search()

## 7. Analytics and Statistics

In [12]:
# Generate analytics about the processed papers
if len(search_engine.papers) > 0:
    df_papers = pd.DataFrame(search_engine.papers)
    
    print("📊 Dataset Analytics")
    print("=" * 40)
    print(f"Total papers: {len(df_papers)}")
    print(f"Recent papers (last 2 years): {df_papers['is_recent'].sum()}")
    print(f"Papers with keywords in title: {df_papers['title_has_keywords'].sum()}")
    
    # Publication year distribution
    df_papers['pub_year'] = pd.to_datetime(df_papers['published']).dt.year
    year_counts = df_papers['pub_year'].value_counts().sort_index()
    
    print("\n📅 Publication Year Distribution (last 5 years):")
    current_year = datetime.now().year
    for year in range(current_year-4, current_year+1):
        count = year_counts.get(year, 0)
        print(f"  {year}: {count} papers")
    
    # Most common words in titles
    all_titles = ' '.join(df_papers['title'].str.lower())
    words = all_titles.split()
    # Filter out common stop words and short words
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'via', 'using', 'based'}
    filtered_words = [word for word in words if len(word) > 2 and word not in stop_words]
    
    from collections import Counter
    word_counts = Counter(filtered_words)
    
    print("\n🏷️ Most Common Words in Titles:")
    for word, count in word_counts.most_common(10):
        print(f"  {word}: {count}")

else:
    print("No papers available for analytics.")

NameError: name 'search_engine' is not defined

## 8. Export and Save Results

In [13]:
# Export results to different formats
if len(search_engine.papers) > 0:
    df_export = pd.DataFrame(search_engine.papers)
    
    # Save as CSV
    csv_filename = f"arxiv_papers_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df_export.to_csv(csv_filename, index=False)
    print(f"✅ Saved {len(df_export)} papers to {csv_filename}")
    
    # Save embeddings separately
    embeddings_filename = f"arxiv_embeddings_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
    with open(embeddings_filename, 'wb') as f:
        pickle.dump({
            'embeddings': search_engine.embeddings,
            'model_name': search_engine.data['model_name'],
            'papers_metadata': search_engine.papers
        }, f)
    print(f"✅ Saved embeddings to {embeddings_filename}")
    
    # Save recent papers as JSON
    recent_papers = df_export[df_export['is_recent']]
    if len(recent_papers) > 0:
        json_filename = f"recent_papers_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        recent_papers.to_json(json_filename, orient='records', indent=2)
        print(f"✅ Saved {len(recent_papers)} recent papers to {json_filename}")

else:
    print("No papers available for export.")

NameError: name 'search_engine' is not defined

## Conclusion

This integrated notebook successfully combines:
- **ArXiv data collection** using the ArXiv API
- **Data processing** with pandas and d6tflow for workflow management
- **Semantic embeddings** using SentenceTransformers
- **Similarity search** for finding relevant papers
- **Analytics and visualization** of the research landscape

The system can be extended with:
- More sophisticated query expansion
- Different embedding models
- Clustering and topic modeling
- Citation network analysis
- Real-time updates and notifications